In [49]:
import os
from collections import Counter
from eval_functions import *

In [50]:
def count_operators(text):
    operators = ['AND', 'OR', 'NOT']
    counts = defaultdict(int)
    positions = {op: [] for op in operators}

    words = text.split()
    for i, word in enumerate(words):
        clean_word = word.strip('[],()')  # Entfernt Klammern und Kommas
        if clean_word in operators:
            counts[clean_word] += 1
            positions[clean_word].append(i)

    return counts, positions

def compare_operators(label_counts, label_positions, model_counts, model_positions):
    comparison = {
        'AND': {'count_match': label_counts['AND'] == model_counts['AND'], 'position_match': label_positions['AND'] == model_positions['AND']},
        'OR': {'count_match': label_counts['OR'] == model_counts['OR'], 'position_match': label_positions['OR'] == model_positions['OR']},
        'NOT': {'count_match': label_counts['NOT'] == model_counts['NOT'], 'position_match': label_positions['NOT'] == model_positions['NOT']}
    }
    return comparison


In [54]:
main_directory = 'model_output'
models = find_folders_with_output(main_directory)
eval_path = f"metrics"
os.makedirs(eval_path, exist_ok=True)
results = []



model_output\Llama-3-70B-Instruct_4_shot
model_output\Llama-3-8B-Instruct_4_shot


In [55]:
for model in [models[0]]:# models:
    p2_label_path = "../../chia_label/p1_model_input"
    ready_path = f"model_output/{model}/output"

    label_files = {extract_nct_number(f): os.path.join(p2_label_path, f) for f in os.listdir(p2_label_path) if f.endswith('.txt')}
    model_files = {extract_nct_number(f): os.path.join(ready_path, f) for f in os.listdir(ready_path) if f.endswith('.txt')}
    common_ncts = set(label_files.keys()).intersection(model_files.keys())

    for nct in common_ncts:
        label_data = read_text(label_files[nct])
        model_data = read_text(model_files[nct])
        label_counts, label_positions = count_operators(label_data)
        model_counts, model_positions = count_operators(model_data)

        comparison = compare_operators(label_counts, label_positions, model_counts, model_positions)
        bleu_score = calculate_bleu(reference=label_data, hypothesis=model_data)
        jaccard_score = jaccard_similarity(label_data, model_data)



        print("-" * 40)
        print(f"NCT: {nct}")
        print(f"Label counts: {dict(label_counts)}")  # Gibt den Zähler als Dictionary aus
        print(f"Model counts: {dict(model_counts)}")
    
        print(label_data)
        print("-" * 20)
        print(model_data)
        print(f"Comparison: {comparison}")
        print(f"BLEU Score: {bleu_score}")
        print(f"Jaccard Similarity: {jaccard_score}")
        print("-" * 40)

        result = {
            'NCT': nct,
            'Model': model,
            'Label AND': label_counts['AND'],
            'Label OR': label_counts['OR'],
            'Label NOT': label_counts['NOT'],
            'Model AND': model_counts['AND'],
            'Model OR': model_counts['OR'],
            'Model NOT': model_counts['NOT'],
            'BLEU Score': bleu_score,
            'Jaccard Similarity': jaccard_score,
            'AND Count Match': comparison['AND']['count_match'],
            'AND Position Match': comparison['AND']['position_match'],
            'OR Count Match': comparison['OR']['count_match'],
            'OR Position Match': comparison['OR']['position_match'],
            'NOT Count Match': comparison['NOT']['count_match'],
            'NOT Position Match': comparison['NOT']['position_match'],
        }
        results.append(result)

df_results = pd.DataFrame(results)

----------------------------------------
NCT: NCT00962364_exc
Label counts: {'AND': 0, 'OR': 0, 'NOT': 0}
Model counts: {'AND': 2, 'OR': 0, 'NOT': 0}
none, all patients meeting the inclusion criteria will be eligible.       

    


--------------------
None, all patients [AND] meeting the inclusion criteria [AND] will be eligible.
Comparison: {'AND': {'count_match': False, 'position_match': False}, 'OR': {'count_match': True, 'position_match': True}, 'NOT': {'count_match': True, 'position_match': True}}
BLEU Score: 0.33085484667982556
Jaccard Similarity: 0.7692307692307693
----------------------------------------
----------------------------------------
NCT: NCT00305097_inc
Label counts: {'AND': 0, 'OR': 0, 'NOT': 0}
Model counts: {'AND': 4, 'NOT': 1, 'OR': 0}
Aged at least 18 years with an ability and willingness to give written informed consent. 

Body mass index 25-35 kg/m2 

Users of at least 2 cups of caffeinated coffee per day who are willing to be randomized to any of the inter

C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnin

----------------------------------------
NCT: NCT00730301_exc
Label counts: {'OR': 14, 'NOT': 1, 'AND': 1}
Model counts: {'OR': 15, 'AND': 2, 'NOT': 0}
Prior endobronchial treatment for emphysema 
Pleural [OR] or interstitial disease that [NOT] precludes surgery. 
Prior lung transplant [OR], LVRS [OR], median sternotomy [OR], bullectomy [OR] or lobectomy. 
Clinically significant bronchiectasis 
Pulmonary nodule requiring surgery 
History of recurrent respiratory infections (> 3 hospitalization in the last year) 
Clinically significant (> 4 Tablespoons per day) sputum production 
Fever [OR], elevated white cell count [OR], or other evidence of active infection 
Dysrhythmia [AND] that might pose a risk during [OR] exercise or training 
Congestive heart failure within 6 mo and LVEF < 45% 
Evidence [OR] or history of Cor Pulmonale 
Resting bradycardia [OR] (< 50 beats/min), frequent multifocal PVCs [OR], complex ventricular arrhythmia [OR], sustained SVT 
History of exercise-related syncop

In [56]:
df_results

,NCT,Model,Label AND,Label OR,Label NOT,Model AND,Model OR,Model NOT,BLEU Score,Jaccard Similarity,AND Count Match,AND Position Match,OR Count Match,OR Position Match,NOT Count Match,NOT Position Match
0,NCT00962364_exc,Llama-3-70B-Instruct_4_shot,0,0,0,2,0,0,0.330855,0.769231,False,False,True,True,True,True
1,NCT00305097_inc,Llama-3-70B-Instruct_4_shot,0,0,0,4,0,1,0.609741,0.883721,False,False,True,True,False,False
2,NCT01639664_inc,Llama-3-70B-Instruct_4_shot,1,0,0,4,0,0,0.354156,1.000000,False,False,True,True,True,True
3,NCT00812344_exc,Llama-3-70B-Instruct_4_shot,0,2,0,0,2,0,0.238566,0.783784,True,True,True,True,True,True
4,NCT01184638_exc,Llama-3-70B-Instruct_4_shot,1,0,0,2,2,0,0.241473,0.756757,False,False,False,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
291,NCT01205334_inc,Llama-3-70B-Instruct_4_shot,3,1,1,15,2,1,0.571173,1.000000,False,False,False,False,True,False
292,NCT00720031_exc,Llama-3-70B-Instruct_4_shot,0,9,1,7,5,2,0.492064,0.953488,False,False,False,False,False,False
293,NCT01717911_exc,Llama-3-70B-Instruct_4_shot,2,3,0,1,2,1,0.550154,0.963636,False,False,False,False,False,False
294,NCT00728156_exc,Llama-3-70B-Instruct_4_shot,2,5,0,3,7,0,0.753889,0.957447,False,False,False,False,True,True


### Model Output to nice JSON and Failure 

In [ ]:
for model in models:
    p2_label_path = "../../chia_label/p1_model_input"
    ready_path = f"model_output/{model}/output"
    
    label_files = {extract_nct_number(f): os.path.join(p2_label_path, f) for f in os.listdir(p2_label_path) if f.endswith('.txt')}
    model_files = {extract_nct_number(f): os.path.join(ready_path, f) for f in os.listdir(ready_path) if f.endswith('.txt')}
    common_ncts = set(label_files.keys()).intersection(model_files.keys())
    labels = []
    predictions = []
    success_data = []
    
    for nct in common_ncts:
    
        label_data = read_text(label_files[nct])
        model_data = read_text(model_files[nct])
            
        
        bleu_score = calculate_bleu(reference=label_data, hypothesis=model_data)
        jaccard_score = jaccard_similarity(label_data, model_data)
  
        print(f"BLEU Score: {bleu_score}")
        print(f"Jaccard Similarity: {jaccard_score}")
        



    
